# PA5 — Enterprise-Grade Agentic AI Game Development
### Roll Number: 25280041
---


## Cell 1 — Install Dependencies

In [0]:
!pip install langgraph langchain langchain-core mlflow tiktoken
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import re
import ast
import json
import time
import mlflow
import tiktoken
import subprocess
from typing import TypedDict, Annotated, List, Any
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
print('Imports done')

Imports done


In [0]:
import mlflow.deployments
client = mlflow.deployments.get_deploy_client('databricks')

def call_llm(system: str, user: str, max_tokens: int = 4096) -> str:
    response = client.predict(
        endpoint='databricks-meta-llama-3-3-70b-instruct',
        inputs={
            'messages': [
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': user}
            ],
            'max_tokens': max_tokens
        }
    )
    return response['choices'][0]['message']['content']

print('LLM client ready')

LLM client ready


In [0]:
class GameState(TypedDict):
    director_messages:  Annotated[List[BaseMessage], add_messages]
    architect_messages: Annotated[List[BaseMessage], add_messages]
    engineer_code:      Annotated[List[str], lambda x, y: x + y]
    qa_feedback:        Annotated[List[str], lambda x, y: x + y]
    current_actor:      str
    iteration:          int
    iteration_score:    Annotated[List[float], lambda x, y: x + y]
    file_saved:         bool
    run_output:         str
    hitl_feedback:      str          # custom feedback injected by human
    pii_report:         dict         # guardrail redaction report
    memory_summary:     str          # summarised context when tokens exceed threshold
    mlflow_run_id:      str

print('GameState defined')

GameState defined


In [0]:
PII_PATTERNS = {
    'email':   (r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[REDACTED_EMAIL]'),
    'api_key': (r'(?i)(sk-|key-|token-|bearer\s)[a-zA-Z0-9\-_]{6,}', '[REDACTED_API_KEY]'),
    'password':(r'(?i)(password|passwd|pwd)\s*[=:]\s*\S+',           '[REDACTED_PASSWORD]'),
}

def pii_middleware(text: str) -> tuple[str, dict]:
    """
    Scans text for PII and redacts it.
    Returns (cleaned_text, report_dict).
    """
    report = {'redactions': [], 'clean': True}
    for pii_type, (pattern, replacement) in PII_PATTERNS.items():
        matches = re.findall(pattern, text)
        if matches:
            report['clean'] = False
            report['redactions'].append({
                'type':  pii_type,
                'count': len(matches),
                'replaced_with': replacement
            })
            text = re.sub(pattern, replacement, text)
    return text, report

_sample = 'My email is dev@example.com and my key is sk-abc123XYZ password=secret123'
_clean, _rep = pii_middleware(_sample)
print('PII test input :', _sample)
print('PII test output:', _clean)
print('Report         :', json.dumps(_rep, indent=2))

with open('guardrail_report.json', 'w') as f:
    json.dump({'sample_input': _sample, 'redacted_output': _clean, 'report': _rep}, f, indent=2)
print(' guardrail_report.json saved')

PII test input : My email is dev@example.com and my key is sk-abc123XYZ password=secret123
PII test output: My email is [REDACTED_EMAIL] and my key is [REDACTED_API_KEY] [REDACTED_PASSWORD]
Report         : {
  "redactions": [
    {
      "type": "email",
      "count": 1,
      "replaced_with": "[REDACTED_EMAIL]"
    },
    {
      "type": "api_key",
      "count": 1,
      "replaced_with": "[REDACTED_API_KEY]"
    },
    {
      "type": "password",
      "count": 1,
      "replaced_with": "[REDACTED_PASSWORD]"
    }
  ],
  "clean": false
}
 guardrail_report.json saved


In [0]:
TOKEN_THRESHOLD = 3000   

def count_tokens(text: str) -> int:
    """Approximate token count using tiktoken cl100k_base."""
    try:
        enc = tiktoken.get_encoding('cl100k_base')
        return len(enc.encode(text))
    except Exception:
        return len(text.split())  

def summarise_if_needed(state: GameState) -> dict:
    """
    Checks combined token count of architect history + QA feedback.
    If above threshold, replaces them with a concise LLM summary.
    Returns a dict of updated fields (empty dict = no change needed).
    """
    arch_text = ' '.join([m.content for m in state.get('architect_messages', [])])
    qa_text   = ' '.join(state.get('qa_feedback', []))
    combined  = arch_text + ' ' + qa_text
    tokens    = count_tokens(combined)

    print(f'  [Memory] Current token count: {tokens} / {TOKEN_THRESHOLD}')

    if tokens < TOKEN_THRESHOLD:
        return {}

    print('  [Memory] Threshold exceeded — summarising...')
    summary = call_llm(
        system='You are a concise technical summariser. Preserve only critical design decisions and bug fixes.',
        user=f"""Summarise the following game development history into ≤200 words.
Retain: architect design decisions, confirmed bugs, features that passed QA, and current score.

ARCHITECT HISTORY:
{arch_text[-2000:]}

QA FEEDBACK HISTORY:
{qa_text[-2000:]}""",
        max_tokens=512
    )
    print(f'  [Memory] Summary ({count_tokens(summary)} tokens): {summary[:200]}...')
    return {'memory_summary': summary}

print('Summarisation middleware defined')

Summarisation middleware defined


In [0]:
def code_interpreter_tool(code: str, filename: str = '25280041_dino_runner.py') -> dict:
    """
    CodeInterpreterTool: saves code to disk and runs a syntax check via ast.parse.
    Returns {'success': bool, 'error': str or None, 'output': str}.
    Primary tool: ast.parse (no __pycache__ needed — works in Databricks).
    Fallback: subprocess py_compile (if ast fails unexpectedly).
    """
    
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(code)
    except Exception as e:
        return {'success': False, 'error': f'File write failed: {e}', 'output': ''}

    try:
        ast.parse(code)
        return {'success': True, 'error': None, 'output': 'Syntax OK (ast.parse)'}
    except SyntaxError as e:
        primary_error = f'SyntaxError at line {e.lineno}: {e.msg} — {e.text}'

    try:
        result = subprocess.run(
            ['python', '-c', f'import py_compile; py_compile.compile("{filename}", doraise=True)'],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            return {'success': True, 'error': None, 'output': 'Syntax OK (fallback py_compile)'}
        else:
            return {'success': False, 'error': result.stderr, 'output': ''}
    except Exception as e:
        return {'success': False, 'error': primary_error, 'output': ''}

print('CodeInterpreterTool defined')

CodeInterpreterTool defined


In [0]:
def director_node(state: GameState) -> dict:
    raw_prompt = input('Awaiting Director Prompt: ')

    # PII Guardrail
    clean_prompt, pii_report = pii_middleware(raw_prompt)
    if not pii_report['clean']:
        print(f' PII DETECTED AND REDACTED: {pii_report["redactions"]}')
    else:
        print('No PII detected')

    #  MLflow experiment
    mlflow.set_experiment('/PA5_DinoRunner')
    run = mlflow.start_run(run_name=f'pa5_iteration_0')
    mlflow.log_param('director_prompt_length', len(clean_prompt))
    mlflow.log_param('pii_detected', not pii_report['clean'])
    print(f'  [MLflow] Run started: {run.info.run_id}')

    return {
        'director_messages': [HumanMessage(content=clean_prompt)],
        'current_actor':     'architect',
        'iteration':          0,
        'engineer_code':      [],
        'qa_feedback':        [],
        'iteration_score':    [],
        'file_saved':         False,
        'run_output':         '',
        'hitl_feedback':      '',
        'pii_report':         pii_report,
        'memory_summary':     '',
        'mlflow_run_id':      run.info.run_id
    }

print('Director node defined')

Director node defined


In [0]:
def architect_node(state: GameState) -> dict:
    print(f'\n========== ITERATION {state.get("iteration", 0)} ARCHITECT ==========')
    t0 = time.time()

    director_prompt  = state['director_messages'][-1].content
    hitl_feedback    = state.get('hitl_feedback', '')
    memory_summary   = state.get('memory_summary', '')

    context = ''
    if memory_summary:
        context += f'\n\nPREVIOUS SESSION SUMMARY (condensed):\n{memory_summary}'
    if hitl_feedback:
        context += f'\n\nDIRECTOR REVISION FEEDBACK:\n{hitl_feedback}'

    response = call_llm(
        system='You are a senior software architect. Produce detailed, implementable system designs.',
        user=f"""Design a complete Pygame Dino Runner game based on this request:
{director_prompt}
{context}

Include: game loop, player mechanics, obstacle types (cactus + pterodactyl),
day/night cycle, clustered obstacles, speed scaling, persistent high score, scoring, and rendering."""
    )

    latency = time.time() - t0
    with mlflow.start_run(run_id=state.get('mlflow_run_id', ''), nested=True):
        mlflow.log_metric('architect_latency_sec', latency, step=state.get('iteration', 0))

    print(f'  [Architect] Design ready ({latency:.1f}s)')

    print('\n  [HITL] Review architect design above.')
    approval = input('  Approve architect design? (y/n): ').strip().lower()
    if approval != 'y':
        revision = input('  Enter revision note for architect: ').strip()
        response += f'\n\n[REVISION REQUESTED]: {revision}'

    return {
        'architect_messages': [AIMessage(content=response)],
        'current_actor': 'engineer'
    }

print('Architect node defined')

Architect node defined


In [0]:
MAX_REACT_STEPS = 3   

def engineer_react_agent(state: GameState) -> dict:
    print('\n===== ENGINEER ReAct AGENT =====')
    t0 = time.time()

    architect_message = state['architect_messages'][-1].content
    prev_code         = state['engineer_code'][-1] if state['engineer_code'] else ''
    qa_feedback       = state['qa_feedback'][-1]   if state['qa_feedback']   else ''
    memory_summary    = state.get('memory_summary', '')
    if qa_feedback and prev_code:
        thought = f'Previous code scored low. QA feedback: {qa_feedback[:300]}. I will fix all issues.'
        prompt  = f"""You are an expert Python pygame developer.

PREVIOUS CODE (scored low — fix it):
```python
{prev_code}
```

QA BUGS TO FIX:
{qa_feedback}

{'MEMORY SUMMARY: ' + memory_summary if memory_summary else ''}

Rewrite the COMPLETE fixed Python file. Requirements:
1. Pterodactyl flying obstacles at mid-air height (duck to avoid)
2. Cactus ground obstacles (clustered variants)
3. Gravity-based jumping physics + terminal velocity
4. SPACE/UP to jump, DOWN to duck
5. Collision detection with inset hitboxes
6. Game-over screen + SPACE to restart
7. Score + persistent high score (file-backed)
8. Day/Night cycle (background colour shifts every N points)
9. Speed increases as score rises
Output ONLY raw Python code. No backticks. No explanations."""
    else:
        thought = 'No previous code. Generating fresh implementation from architect design.'
        prompt  = f"""You are an expert Python pygame developer.
Architect design:
{architect_message}

Write a COMPLETE, runnable Pygame Dino Runner. Requirements:
1. Pterodactyl flying obstacles at mid-air height (duck to avoid)
2. Cactus ground obstacles (clustered variants)
3. Gravity-based jumping physics + terminal velocity
4. SPACE/UP to jump, DOWN to duck
5. Collision detection with inset hitboxes
6. Game-over screen + SPACE to restart
7. Score + persistent high score (file-backed)
8. Day/Night cycle (background colour shifts every N points)
9. Speed increases as score rises
Output ONLY raw Python code. No backticks. No explanations."""

    print(f'  [THOUGHT] {thought}')

    code = ''
    tool_result = {}

    for step in range(1, MAX_REACT_STEPS + 1):
        print(f'\n  [ACTION] Step {step}/{MAX_REACT_STEPS} — calling LLM...')
        raw = call_llm(
            system='You are an expert Python pygame developer. Output ONLY raw Python code. Never truncate.',
            user=prompt
        )

        code = raw.strip()
        if code.startswith('```'):
            code = code.split('\n', 1)[1]
        if code.endswith('```'):
            code = code.rsplit('```', 1)[0]
        code = code.strip()

        print(f'  [ACTION] Running CodeInterpreterTool ({len(code)} chars)...')
        tool_result = code_interpreter_tool(code)
        print(f'  [OBSERVATION] Tool result: success={tool_result["success"]} | {tool_result["output"] or tool_result["error"]}')

        if tool_result['success']:
            print(f' Code passed syntax check on step {step}')
            break
        else:
            error = tool_result['error']
            print(f'   Syntax error detected. Self-repairing... ({error[:100]})')
            prompt = f"""Your code has a syntax error:
{error}

Here is the broken code:
```python
{code}
```
Fix ONLY the syntax error and return the complete corrected Python file.
Output ONLY raw Python code. No backticks."""

    latency = time.time() - t0
    with mlflow.start_run(run_id=state.get('mlflow_run_id', ''), nested=True):
        mlflow.log_metric('engineer_latency_sec',   latency,                     step=state.get('iteration', 0))
        mlflow.log_metric('engineer_code_length',   len(code),                   step=state.get('iteration', 0))
        mlflow.log_metric('syntax_check_passed',    int(tool_result.get('success', False)), step=state.get('iteration', 0))

    print('\n  [HITL] Review engineer output above.')
    input('  Press ENTER to continue to QA...')

    return {
        'engineer_code': [code],
        'run_output':    tool_result.get('output', '') or tool_result.get('error', ''),
        'current_actor': 'qa'
    }

print('Engineer ReAct agent defined')

Engineer ReAct agent defined


In [0]:
def qa_react_agent(state: GameState) -> dict:
    print('\n===== QA ReAct AGENT =====')
    t0 = time.time()

    code       = state['engineer_code'][-1]
    run_output = state.get('run_output', '')
    design     = state['architect_messages'][-1].content
    print('  [THOUGHT] I will review the code in three phases: syntax, logic, performance.')

    print('  [ACTION] Phase 1 — Syntax check')
    syntax_report = call_llm(
        system='You are a Python syntax expert. Be brief and specific.',
        user=f"""Check for syntax/import/structural errors in this code.
Ignore pygame display errors (environment limitation).
Run output: {run_output}
Code:
{code}
Reply 'SYNTAX OK' if no issues, else list specific errors with line numbers."""
    )
    print(f'  [OBSERVATION] Syntax: {syntax_report[:100]}')

    print('  [ACTION] Phase 2 — Logic check')
    logic_report = call_llm(
        system='You are a game logic QA specialist. Be concise.',
        user=f"""Check game logic against design. Ignore syntax issues already reported.
Design: {design[:500]}
Code:
{code}
For each: PASS/FAIL/PARTIAL — jumping, ducking, cacti, pterodactyls, collision, game-over, score, day/night, speed scaling."""
    )
    print(f'  [OBSERVATION] Logic: {logic_report[:100]}')


    print('  [ACTION] Phase 3 — Performance + score')
    perf_report = call_llm(
        system='You are a performance auditor.',
        user=f"""Audit for: FPS cap, off-screen cleanup, named constants, code structure.
Rate each GOOD/ACCEPTABLE/POOR.
Code snippet:
{code[:1500]}"""
    )

    score_raw = call_llm(
        system='Reply with ONLY a single decimal number 1.0-10.0. Nothing else.',
        user=f"""Score this Pygame Dino Runner code (1-10).
9-10: All features, no bugs, clean
7-8:  Core works, minor issues
5-6:  Runs, missing 2-3 features
3-4:  Multiple failures
1-2:  Broken
Ignore pygame display/video errors.
Syntax: {syntax_report[:300]}
Logic:  {logic_report[:300]}
Perf:   {perf_report[:300]}"""
    )

    try:
        score = float(score_raw.strip().split()[0])
        score = max(1.0, min(10.0, score))
    except Exception:
        score = 5.0

    final_report = f"""=== QA ReAct REPORT (Iteration {state.get('iteration',0)}) ===

[SYNTAX]\n{syntax_report}

[LOGIC]\n{logic_report}

[PERFORMANCE]\n{perf_report}

[SCORE] {score}/10"""

    latency = time.time() - t0
    print('  [MLflow] Computing groundedness score...')
    groundedness_raw = call_llm(
        system='Reply with ONLY a decimal 0.0-1.0.',
        user=f"""Rate how accurately this code reflects the architect design (0=not at all, 1=perfectly).
Design (first 400 chars): {design[:400]}
Code (first 400 chars): {code[:400]}"""
    )
    try:
        groundedness = float(groundedness_raw.strip().split()[0])
        groundedness = max(0.0, min(1.0, groundedness))
    except Exception:
        groundedness = 0.5

    iteration = state.get('iteration', 0)
    with mlflow.start_run(run_id=state.get('mlflow_run_id', ''), nested=True):
        mlflow.log_metric('qa_latency_sec',   latency,      step=iteration)
        mlflow.log_metric('qa_score',         score,        step=iteration)
        mlflow.log_metric('groundedness',     groundedness, step=iteration)

    print(f'  [OBSERVATION] Score={score}/10 | Groundedness={groundedness:.2f} | Latency={latency:.1f}s')

    print('\n  [HITL] Review QA report above.')
    input('  Press ENTER to continue to scorer...')

    return {
        'qa_feedback':     [final_report],
        'iteration_score': [score],
        'current_actor':   'scorer'
    }

print(' QA ReAct agent defined')

 QA ReAct agent defined


In [0]:
def score_node(state: GameState) -> dict:
    score     = state['iteration_score'][-1]
    iteration = state.get('iteration', 0) + 1
    print(f'\n[Score]: {score}/10  |  Iteration: {iteration}')


    memory_updates = summarise_if_needed(state)

    with mlflow.start_run(run_id=state.get('mlflow_run_id', ''), nested=True):
        mlflow.log_metric('iteration_score', score, step=iteration)

    result = {
        'iteration':      iteration,
        'current_actor':  'human_check'
    }
    result.update(memory_updates)
    return result


def should_continue(state: GameState) -> str:
    """
    Task 2.2 HITL: mandatory human decision after scorer.
    - 'done'      → end workflow
    - 'engineer'  → loop back to engineer (quick fix)
    - 'architect' → inject custom feedback, restart from architect
    Minimum 3 iterations enforced (Task 3.1 constraint).
    """
    score     = state['iteration_score'][-1]
    iteration = state.get('iteration', 0)

    print(f'\n========== ITERATION {iteration} COMPLETE ==========')
    print(f'Score: {score}/10')
    print(f'QA Summary (first 400 chars):\n{state["qa_feedback"][-1][:400]}')

    if iteration < 3:
        print(f'\n  [CONSTRAINT] Minimum 3 iterations required. Forcing continuation (iteration {iteration}/3).')
        return 'engineer'

    print('\n  [HITL] Options:')
    print('    d = done (accept code)')
    print('    e = loop to engineer (quick fix)')
    print('    a = inject feedback → architect (full revision)')
    choice = input('  Your choice (d/e/a): ').strip().lower()

    if choice == 'd':
        print('Workflow accepted by director.')
        mlflow.end_run()
        return 'end'
    elif choice == 'a':
        feedback = input('  Enter revision feedback for architect: ').strip()
        state['hitl_feedback'] = feedback
        print('  [Routing] → Architect with revision feedback')
        return 'architect'
    else:
        print('  [Routing] → Engineer for quick fix')
        return 'engineer'

print(' Scorer and should_continue defined')

 Scorer and should_continue defined


In [0]:
memory  = MemorySaver()  
builder = StateGraph(GameState)

builder.add_node('director',  director_node)
builder.add_node('architect', architect_node)
builder.add_node('engineer',  engineer_react_agent)
builder.add_node('qa',        qa_react_agent)
builder.add_node('scorer',    score_node)

builder.add_edge(START,       'director')
builder.add_edge('director',  'architect')
builder.add_edge('architect', 'engineer')
builder.add_edge('engineer',  'qa')
builder.add_edge('qa',        'scorer')

builder.add_conditional_edges(
    'scorer',
    should_continue,
    {
        'engineer':  'engineer',
        'architect': 'architect',
        'end':        END
    }
)

graph = builder.compile(checkpointer=memory)
print('Parent graph compiled with MemorySaver checkpointing')

Parent graph compiled with MemorySaver checkpointing


In [0]:
config = {'configurable': {'thread_id': 'pa5_session_1'}}

initial_state = {
    'director_messages':  [],
    'architect_messages': [],
    'engineer_code':      [],
    'qa_feedback':        [],
    'current_actor':      'director',
    'iteration':           0,
    'iteration_score':    [],
    'file_saved':          False,
    'run_output':          '',
    'hitl_feedback':       '',
    'pii_report':          {},
    'memory_summary':      '',
    'mlflow_run_id':       ''
}

print('--- STARTING PA5 WORKFLOW ---')
for event in graph.stream(initial_state, config=config):
    for node, value in event.items():
        print(f'\n===== {node.upper()} =====')

--- STARTING PA5 WORKFLOW ---


Awaiting Director Prompt:  Build a complete, fully playable Chrome Dino Runner game in a single self-contained Python file using Pygame.  WINDOW & TECHNICAL SETUP: - Window size: 900x400 pixels - 60 FPS cap using pygame.time.Clock().tick(60) - All magic numbers stored as named constants at the top of the file - No external assets required — draw everything using pygame.draw shapes - Single file, no imports beyond pygame, sys, random  PLAYER — DINOSAUR: - Starts at x=100, runs automatically (player controls jump and duck only) - SPACE or UP ARROW: jump using real physics — apply negative velocity upward, then add GRAVITY every frame until grounded - DOWN ARROW: duck — reduce hitbox height to half, snap y to ground level - Cannot jump while already airborne - Landing must snap exactly to ground level, no floating or sinking - Terminal velocity: cap falling speed at 14 pixels/frame  GROUND OBSTACLES — CACTI: - Spawn from right edge, move left at current game speed - Two variants: single c

No PII detected
  [MLflow] Run started: 8fa897b2a7344599abadb47d152855aa

===== DIRECTOR =====

========== ITERATION 0 ARCHITECT ==========
  [Architect] Design ready (29.9s)

  [HITL] Review architect design above.


  Approve architect design? (y/n):  y


===== ARCHITECT =====

===== ENGINEER ReAct AGENT =====
  [THOUGHT] No previous code. Generating fresh implementation from architect design.

  [ACTION] Step 1/3 — calling LLM...
  [ACTION] Running CodeInterpreterTool (8162 chars)...
  [OBSERVATION] Tool result: success=True | Syntax OK (ast.parse)
 Code passed syntax check on step 1

  [HITL] Review engineer output above.


...


===== ENGINEER =====

===== QA ReAct AGENT =====
  [THOUGHT] I will review the code in three phases: syntax, logic, performance.
  [ACTION] Phase 1 — Syntax check
  [OBSERVATION] Syntax: SYNTAX OK
  [ACTION] Phase 2 — Logic check
  [OBSERVATION] Logic: 1. **Jumping**: PASS
   - The player's vertical velocity is set to JUMP_VEL when the space or up key
  [ACTION] Phase 3 — Performance + score
  [MLflow] Computing groundedness score...
  [OBSERVATION] Score=8.0/10 | Groundedness=1.00 | Latency=10.8s

  [HITL] Review QA report above.


  Press ENTER to continue to scorer... 


===== QA =====

[Score]: 8.0/10  |  Iteration: 3
  [Memory] Current token count: 4618 / 3000
  [Memory] Threshold exceeded — summarising...
  [Memory] Summary (202 tokens): Here's a concise summary of the game development history in ≤200 words:

**Architect Design Decisions:**
- The game runs at 60 FPS with a window size of 900x400 pixels.
- The player can jump and duck ...

========== ITERATION 3 COMPLETE ==========
Score: 8.0/10
QA Summary (first 400 chars):
=== QA ReAct REPORT (Iteration 2) ===

[SYNTAX]
SYNTAX OK

[LOGIC]
1. **Jumping**: PASS
   - The player's vertical velocity is set to JUMP_VEL when the space or up key is pressed and the player is on the ground.
   - The player's vertical velocity is updated based on gravity.

2. **Ducking**: PASS
   - The player's ducking status is toggled when the down key is pressed or released.
   - The player

  [HITL] Options:
    d = done (accept code)
    e = loop to engineer (quick fix)
    a = inject feedback → architect (full revision

  Your choice (d/e/a):  d

Workflow accepted by director.

===== SCORER =====
